## Task 3: Wrapper Methods

### Subtask 1: Use Sequential Forward Selection (SFS) and Sequential Backward Selection (SBS) to obtain a maximum of 40 features in each case. Plot the RMSE against the number of features for both SFS and SBS, and use the shape of curves to justify your choise of the number of features.

In [ ]:
# Prepare cross validation folds 
cv = KFold(n_splits = 5, shuffle = True, random_state = 4)

folds = []
feature_names_cv = None # variable to store encoded feature names 

for train_idx, validation_idx in cv.split(X_train):
    preprocessing = clone(preprocessing_pipeline) # use fresch copy of task 1 preprocessing
    X_train_subset = preprocessing.fit_transform(X_train.iloc[train_idx]) # fit preprocessing on training fold
    X_validation_subset = preprocessing.transform(X_train.iloc[validation_idx]) # transform validation fold using same preprocessing on X_tr

    # get features names produced via encoding and then assign them 
    names = list(preprocessing.named_steps["encoding"].get_feature_names_out())

    if feature_names_cv is None:
        feature_names_cv = names
    else:
        assert names == feature_names_cv

    folds.append(( 
        X_train_subset,
        X_validation_subset,
        y_train.iloc[train_idx],
        y_train.iloc[validation_idx]
    ))

model = MLPRegressor(
    hidden_layer_sizes = (100, 50),
    max_iter = 5000,
    random_state = 4
)

In [ ]:
# the steps to implement SFS and SBS where taken directly from the notes
# SFS:
# Step 1: Start with an empty feature set 
selected_features = []
sfs_results = []

# Step 6: Repeat steps 2 to 5 until the stopping condition is reached (40 features selected)
while len(selected_features) < 40: 
    scores = {}

    # Step 4: Repeat steps 2 and 3 for all features not already in the set
    for feature in range(len(feature_names_cv)): 
        if feature in selected_features: continue

        # step 2: Temporarily add a feature to the set and train the model 
        candidate_subset = selected_features + [feature]
        fold_rmses = []

        for X_train_subset, X_validation_subset, y_train_subset, y_validation_subset in folds:
            fold_model = clone(model)
            fold_model.fit(X_train_subset[:, candidate_subset], y_train_subset)

            # Step 3: Compute the evaluation metric and store it.
            predictions = fold_model.predict(X_validation_subset[:, candidate_subset])
            rmse = np.sqrt(mean_squared_error(y_validation_subset, predictions))

            fold_rmses.append(rmse)

        scores[feature] = np.mean(fold_rmses)

    # Step 5: Check which added feature gave the best evaluation metric and add it to the set permanently.
    best_feature = min(scores, key = scores.get)
    selected_features.append(best_feature)

    sfs_results.append((
        len(selected_features),
        scores[best_feature],
        selected_features.copy()
    ))

    print("SFS:", len(selected_features), "features, RMSE:", scores[best_feature])

/home/dave/miniconda3/envs/cps3235/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:792: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


In [ ]:
# SBS:
# Step 1: Start with a full feature set 
selected_features = list(range(len(feature_names_cv)))
sbs_results = []

# Step 6: Repeat steps 2 to 5 until the stopping condition is reached (1 feature left)
while len(selected_features) > 1: 
    scores = {}

    # Step 4: Repeat steps 2 and 3 for all features already in the set
    for feature in selected_features: 

        # step 2: Temporarily remove a feature from the set and train the model 
        candidate_subset = [f for f in selected_features if f != feature]
        fold_rmses = []

        for X_train_subset, X_validation_subset, y_train_subset, y_validation_subset in folds:
            fold_model = clone(model)
            fold_model.fit(X_train_subset[:, candidate_subset], y_train_subset)

            # Step 3: Compute the evaluation metric and store it.
            predictions = fold_model.predict(X_validation_subset[:, candidate_subset])
            rmse = np.sqrt(mean_squared_error(y_validation_subset, predictions))

            fold_rmses.append(rmse)

        scores[feature] = np.mean(fold_rmses)

    # Step 5: Check which removed feature gave the best evaluation metric and remove it from the set permanently.
    best_feature = min(scores, key = scores.get)
    selected_features.remove(best_feature)

    if len(selected_features) <= 40:
        sbs_results.append((
            len(selected_features),
            scores[best_feature],
            selected_features.copy()
        ))

    print("SBS:", len(selected_features), "features, RMSE:", scores[best_feature])

In [ ]:
# Plot the RMSE against the number of features for both SFS and SBS
plt.figure(figsize=(10, 6))

plt.plot(
    [r[0] for r in sfs_results],
    [r[1] for r in sfs_results],
    label="SFS"
)

plt.plot(
    [r[0] for r in sbs_results],
    [r[1] for r in sbs_results],
    label="SBS"
)

plt.xlabel("Number of Features")
plt.ylabel("Mean 5-Fold CV RMSE")
plt.title("SFS vs SBS")
plt.legend()
plt.grid()
plt.show()